<h1>Vorbereitung</h1>

In [11]:
import os
import tarfile
import urllib.request

DOWNLOAD_ROOT = "https://raw.githubusercontent.com/ageron/handson-ml/master/"
HOUSING_PATH = "datasets/housing"
HOUSING_URL = DOWNLOAD_ROOT + HOUSING_PATH + "/housing.tgz"

def fetch_housing_data(housing_url=HOUSING_URL, housing_path=HOUSING_PATH):
    os.makedirs(housing_path, exist_ok=True)
    tgz_path = os.path.join(housing_path, "housing.tgz")
    urllib.request.urlretrieve(housing_url, tgz_path)
    
    housing_tgz = tarfile.open(tgz_path)
    housing_tgz.extractall(path=housing_path)
    housing_tgz.close()

fetch_housing_data()

C:\Users\roger\AppData\Local\Temp\ipykernel_32868\3151530550.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  housing_tgz.extractall(path=housing_path)


In [13]:
import pandas as pd

housing = pd.read_csv("datasets/housing/housing.csv")
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [15]:
housing = pd.read_csv("datasets/housing/housing.csv")

X = housing.drop("median_house_value", axis=1)
y = housing["median_house_value"]

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

num_attribs = list(X_train.drop("ocean_proximity", axis=1))
cat_attribs = ["ocean_proximity"]

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(), cat_attribs)
])

In [21]:
X_train_prepared = full_pipeline.fit_transform(X_train)
X_test_prepared = full_pipeline.transform(X_test)

<h1>Aufgabe 1: </h1>
<p>1. Probieren Sie einen Support Vector Machine Regressor
(sklearn.svm.SVR) mit unterschiedlichen Hyperparametern aus,
beispielsweise kernel="linear" (mit unterschiedlichen Werten für
den Hyperparameter C) oder kernel="rbf" (mit unterschiedlichen
Werten für die Hyperparameter C und gamma). Kümmern Sie sich im
Moment nicht zu sehr darum, was Hyperparameter überhaupt sind. Wie gut
schneidet der beste SVR-Prädiktor ab?</p>

<h2>Schritt 1 - Grid Search vorbereiten</h2>

In [23]:
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error
import numpy as np

param_grid = [
    {"kernel": ["linear"], "C": [10, 30, 100, 300, 1000]},
    {"kernel": ["rbf"], "C": [1, 3, 10, 30, 100, 300, 1000], "gamma": [0.01, 0.03, 0.1, 0.3, 1.0]}
]

svr = SVR()

grid_search = GridSearchCV(
    svr,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train_prepared, y_train)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


GridSearchCV(cv=5, estimator=SVR(), n_jobs=-1,
             param_grid=[{'C': [10, 30, 100, 300, 1000], 'kernel': ['linear']},
                         {'C': [1, 3, 10, 30, 100, 300, 1000],
                          'gamma': [0.01, 0.03, 0.1, 0.3, 1.0],
                          'kernel': ['rbf']}],
             scoring='neg_mean_squared_error', verbose=2)

<h2>Schritt 2 - Bestes Modell anzeigen</h2>

In [25]:
grid_search.best_params_

{'C': 1000, 'kernel': 'linear'}

In [27]:
grid_search.best_estimator_

SVR(C=1000, kernel='linear')

<h2>Schritt 3 - Leistung auf dem Trainings-Set als RMSE anschauen</h2>

In [29]:
best_svr_model = grid_search.best_estimator_

housing_predictions = best_svr_model.predict(X_train_prepared)
svr_mse = mean_squared_error(y_train, housing_predictions)
svr_rmse = np.sqrt(svr_mse)

svr_rmse

69889.21670798874

<h2>Schritt 4 - Leistung auf dem Test-Set prüfen</h2>

In [31]:
X_test_prepared = full_pipeline.transform(X_test)

In [33]:
best_svr_model = grid_search.best_estimator_

<h2>Finale Bewertung</h2>

In [35]:
from sklearn.metrics import mean_squared_error
import numpy as np

final_predictions = best_svr_model.predict(X_test_prepared)
final_mse = mean_squared_error(y_test, final_predictions)
final_rmse = np.sqrt(final_mse)

final_rmse

71420.51912783133

<p>Es wurde ein Support Vector Regressor (SVR) mit verschiedenen Hyperparametern getestet. Dabei wurden sowohl ein linearer Kernel mit unterschiedlichen Werten für C als auch ein RBF-Kernel mit unterschiedlichen Werten für C und gamma untersucht. Die Hyperparameter wurden mithilfe von GridSearchCV optimiert.

Für die Datenvorverarbeitung wurde eine Pipeline verwendet, die fehlende Werte mit dem Median ersetzt, numerische Attribute skaliert und das kategoriale Attribut mittels One-Hot-Encoding transformiert.

Das beste Modell verwendete einen linearen Kernel mit C = 1000. Die Leistung wurde auf dem Testdatensatz anhand des RMSE (Root Mean Squared Error) bewertet. Der beste SVR-Prädiktor erreichte einen RMSE von ca. 71'420.

Im Vergleich zu anderen Modellen zeigt sich, dass der SVR auf diesem Datensatz eher schwächer abschneidet.</p>

<h1>Aufgabe 2</h1>
<p>Ersetzen Sie GridSearchCV durch RandomizedSearchCV.</p>

<h2>Schritt 1- RandomizedSearch vorbereiten</h2>

In [37]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
import numpy as np

param_distributions = {
    "kernel": ["linear", "rbf"],
    "C": [1, 3, 10, 30, 100, 300, 1000],
    "gamma": [0.01, 0.03, 0.1, 0.3, 1.0]
}

svr = SVR()

rnd_search = RandomizedSearchCV(
    svr,
    param_distributions=param_distributions,
    n_iter=10,
    cv=5,
    scoring="neg_mean_squared_error",
    verbose=2,
    n_jobs=-1,
    random_state=42
)

rnd_search.fit(X_train_prepared, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=5, estimator=SVR(), n_jobs=-1,
                   param_distributions={'C': [1, 3, 10, 30, 100, 300, 1000],
                                        'gamma': [0.01, 0.03, 0.1, 0.3, 1.0],
                                        'kernel': ['linear', 'rbf']},
                   random_state=42, scoring='neg_mean_squared_error',
                   verbose=2)

<h2>Schritt 2 - Bestes Modell anzeigen</h2>

In [39]:
rnd_search.best_params_

{'kernel': 'linear', 'gamma': 0.1, 'C': 300}

In [41]:
best_svr_random = rnd_search.best_estimator_

<h2>Schritt 3 - Auf Tesdaten bewerten</h2>

In [43]:
final_predictions_rnd = best_svr_random.predict(X_test_prepared)
final_mse_rnd = mean_squared_error(y_test, final_predictions_rnd)
final_rmse_rnd = np.sqrt(final_mse_rnd)

final_rmse_rnd

71468.10310149519

<p>GridSearchCV wurde durch RandomizedSearchCV ersetzt, um nur eine zufällige Auswahl von Hyperparameter-Kombinationen zu testen. Dabei wurden verschiedene Werte für kernel, C und gamma untersucht.

Das beste gefundene Modell verwendete einen linearen Kernel mit 𝐶=300 und 𝛾 =0.1. Die Leistung wurde auf dem Testdatensatz anhand des RMSE bewertet und ergab einen Wert von ca. 71'468.

Im Vergleich zu GridSearchCV liefert RandomizedSearchCV ein ähnlich gutes Ergebnis, ist jedoch effizienter, da nicht alle möglichen Kombinationen getestet werden müssen.</p>

<h1>Aufgabe 3</h1>
<p>Fügen Sie der vorbereitenden Pipeline einen Transformer hinzu, um nur die
wichtigsten Attribute auszuwählen.</p>

<h2>Schritt 1 - Eigenen Transformer definieren</h2>

In [45]:
from sklearn.base import BaseEstimator, TransformerMixin

class TopFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, feature_indices):
        self.feature_indices = feature_indices

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[:, self.feature_indices]

<h2>Schritt 2 - Wichtige Features festlegen</h2>

In [47]:
top_feature_indices = [0, 1, 2, 3, 4, 5, 6, 7] #ersten 8 Features von insgesamt 13

<h2>Schritt 3 - Neue Pipeline mit Feature-Auswahl</h2>

In [49]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR

prepare_select_and_predict_pipeline = Pipeline([
    ("preparation", full_pipeline),
    ("feature_selection", TopFeatureSelector(top_feature_indices)),
    ("svr", SVR(kernel="linear", C=1000))
])

<h2>Schritt 4 - Modell trainieren</h2>

In [51]:
prepare_select_and_predict_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preparation',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['longitude', 'latitude',
                                                   'housing_median_age',
                                                   'total_rooms',
                                                   'total_bedrooms',
                                                   'population', 'households',
                                                   'median_income']),
                                                 ('cat', OneHotEncoder(),
                                                  ['ocean_proximity'])])),
                ('feature_selection',
                 TopFeatureSelector(feature_indices=[0, 1, 2, 3, 4, 5, 6, 7])),
                ('svr', SVR(C=1000, kernel='linear'))])

<h2>Schritt 5 - Auf Testdaten auswerten</h2>

In [53]:
from sklearn.metrics import mean_squared_error
import numpy as np

selected_predictions = prepare_select_and_predict_pipeline.predict(X_test)
selected_mse = mean_squared_error(y_test, selected_predictions)
selected_rmse = np.sqrt(selected_mse)

selected_rmse

72523.40440347696

<p>Es wurde ein eigener Transformer (TopFeatureSelector) implementiert, der nach der Datenvorverarbeitung eine Auswahl bestimmter Attribute trifft. Der Transformer gibt nur ausgewählte Feature-Spalten an das Modell weiter.

Anschliessend wurde eine Pipeline erstellt, die aus der Datenvorverarbeitung, der Feature-Auswahl und einem SVR-Modell besteht. Dadurch wird sichergestellt, dass alle Verarbeitungsschritte konsistent und reproduzierbar ausgeführt werden.

Das Modell wurde anschliessend auf dem Testdatensatz evaluiert. Der resultierende RMSE beträgt ca. 72'523. Im Vergleich zu den vorherigen Modellen zeigt sich, dass die Feature-Auswahl in diesem Fall zu einer leicht schlechteren Leistung führt, was darauf hindeutet, dass möglicherweise wichtige Informationen verloren gegangen sind.</p>

<h1>Aufgabe 4</h1>
<p>Erstellen Sie eine einzige Pipeline, die die vollständige Vorverarbeitung der
Daten und die endgültige Vorhersage durchführt.</p>

<h2>Schritt 1 - Vorverarbeitung definieren</h2>

In [55]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR

num_attribs = list(X_train.drop("ocean_proximity", axis=1))
cat_attribs = ["ocean_proximity"]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(), cat_attribs)
])

<h2>Schritt 2 - Alles in eine einzige Pipeline packen</h2>

In [57]:
#bestes Modell aus Aufgabe 1
full_model_pipeline = Pipeline([
    ("preparation", full_pipeline),
    ("svr", SVR(kernel="linear", C=1000))
])

<h2>Schritt 3 - Modell trainieren</h2>

In [59]:
full_model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preparation',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['longitude', 'latitude',
                                                   'housing_median_age',
                                                   'total_rooms',
                                                   'total_bedrooms',
                                                   'population', 'households',
                                                   'median_income']),
                                                 ('cat', OneHotEncoder(),
                                                  ['ocean_proximity'])])),
                ('svr', SVR(C=1000, kernel='linear'))])

<h2>Schritt 4 - Auf Testdaten auswerten</h2>

In [61]:
from sklearn.metrics import mean_squared_error
import numpy as np

pipeline_predictions = full_model_pipeline.predict(X_test)
pipeline_mse = mean_squared_error(y_test, pipeline_predictions)
pipeline_rmse = np.sqrt(pipeline_mse)

pipeline_rmse

71420.51912783133

<p>Es wurde eine einzelne scikit-learn-Pipeline erstellt, die sowohl die Datenvorverarbeitung als auch das Machine-Learning-Modell umfasst. Die Pipeline beinhaltet das Ersetzen fehlender Werte mittels Median, die Skalierung numerischer Attribute sowie das One-Hot-Encoding des kategorialen Attributs.

Anschliessend wird ein Support Vector Regressor (SVR) mit den zuvor bestimmten optimalen Hyperparametern angewendet. Die Pipeline wurde direkt auf den Trainingsdaten trainiert und anschliessend auf dem Testdatensatz evaluiert.

Der resultierende RMSE beträgt ca. 71'420. Das Ergebnis entspricht dem vorherigen Modell, zeigt jedoch, dass durch die Pipeline eine klarere und robustere Struktur erreicht wird, ohne die Modellleistung zu beeinflussen.</p>

<h1>Aufgabe 5</h1>
<p>Probieren Sie einige der Optionen bei der Vorverarbeitung mit
GridSearchCV aus.</p>

<h2>Schritt 1 - Pipeline definieren</h2>

In [63]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error
import numpy as np

num_attribs = list(X_train.drop("ocean_proximity", axis=1))
cat_attribs = ["ocean_proximity"]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer()),
    ("scaler", StandardScaler())
])

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(), cat_attribs)
])

full_model_pipeline = Pipeline([
    ("preparation", full_pipeline),
    ("svr", SVR())
])

<h2>Schritt 2 - Parameter für Grid Search definieren</h2>

In [65]:
param_grid = [
    {
        "preparation__num__imputer__strategy": ["median", "most_frequent"],
        "svr__kernel": ["linear"],
        "svr__C": [10, 100, 300, 1000]
    },
    {
        "preparation__num__imputer__strategy": ["median", "most_frequent"],
        "svr__kernel": ["rbf"],
        "svr__C": [10, 100, 300, 1000],
        "svr__gamma": [0.01, 0.1, 1.0]
    }
]

<h2>Schritt 3 - Grid Search ausführen</h2>

In [67]:
grid_search_pipeline = GridSearchCV(
    full_model_pipeline,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    verbose=2,
    n_jobs=-1
)

grid_search_pipeline.fit(X_train, y_train)

Fitting 5 folds for each of 32 candidates, totalling 160 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preparation',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['longitude',
                                                                          'latitude',
                                                                          'housing_median_age',
                                                                          'total_rooms',
                                                                          'total_bedrooms',
                                                                          'population',
                                                                          'households',
                                                                          'median_income']),
                                                                        ('cat',
                                                                         OneHotEncoder(),
                                                                         ['ocean_proximity'])])),
                                       ('svr', SVR())]),
             n_jobs=-1,
             param_grid=[{'preparation__num__imputer__strategy': ['median',
                                                                  'most_frequent'],
                          'svr__C': [10, 100, 300, 1000],
                          'svr__kernel': ['linear']},
                         {'preparation__num__imputer__strategy': ['median',
                                                                  'most_frequent'],
                          'svr__C': [10, 100, 300, 1000],
                          'svr__gamma': [0.01, 0.1, 1.0],
                          'svr__kernel': ['rbf']}],
             scoring='neg_mean_squared_error', verbose=2)

<h2>Schritt 4 - Bestes Modell anzeigen</h2>

In [69]:
grid_search_pipeline.best_params_

{'preparation__num__imputer__strategy': 'median',
 'svr__C': 1000,
 'svr__kernel': 'linear'}

In [71]:
best_pipeline_model = grid_search_pipeline.best_estimator_

<h2>Schritt 5 - Auf Testdaten auswerten</h2>

In [73]:
pipeline_predictions = best_pipeline_model.predict(X_test)
pipeline_mse = mean_squared_error(y_test, pipeline_predictions)
pipeline_rmse = np.sqrt(pipeline_mse)

pipeline_rmse

71420.51912783133

<p>Es wurde eine vollständige Pipeline aus Datenvorverarbeitung und SVR-Modell mit GridSearchCV optimiert. Dabei wurden nicht nur Modell-Hyperparameter wie kernel, C und gamma untersucht, sondern auch eine Option der Vorverarbeitung, nämlich die Strategie des SimpleImputer.

Das beste Modell verwendete die Imputationsstrategie median, einen linearen Kernel und den Hyperparameter C = 1000. Die Leistung wurde auf dem Testdatensatz mit dem RMSE bewertet und ergab einen Wert von ca. 71'420.52.

Die Aufgabe zeigt, dass sich mit scikit-learn nicht nur das Modell selbst, sondern auch Schritte der Datenvorverarbeitung gemeinsam in einer Pipeline optimieren lassen.</p>